# DefectosCafeVerde — object-level error complementarity audit

Validation-only inference over the completed D0DIRECT, AF2DIRECT, DCWCF1, LIFRPF1, and RAFC1 checkpoints. This notebook does not train, tune, ensemble, or extract test.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import hashlib, importlib, json, os, shutil, subprocess, sys, tarfile
BRANCH='codex/defectoscafeverde-error-complementarity'
WORK=Path('/content'); REPO=WORK/'coffee-bean-detection'
ARCHIVE_NAME='defectoscafeverde-grouped-physical-v1.tar'
EXPECTED_ARCHIVE_BYTES=686474752
EXPECTED_ARCHIVE_SHA='53fb2233f1f0d1c77cb24eca2d720f86e0a16835b8a69f4e8f3176fae1aacef2'
def sha256(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda:stream.read(8*1024*1024),b''): h.update(block)
    return h.hexdigest()
drive_roots=[root for root in [Path('/content/drive/MyDrive'),*Path('/content/drive/.shortcut-targets-by-id').glob('*')] if root.exists()]
d0_relative=Path('experiments/defectoscafeverde-cwcf-direct-v1/val_reports/D0DIRECT_seed42_result.json')
d0_candidates=sorted({path for root in drive_roots for path in root.rglob(d0_relative.name) if path.as_posix().endswith(d0_relative.as_posix())})
if len(d0_candidates)!=1: raise FileNotFoundError(f'Harus ada tepat satu D0 result; ditemukan {d0_candidates}')
PROJECT=d0_candidates[0].parents[3]
RESULT_REL={
 'D0DIRECT':'experiments/defectoscafeverde-cwcf-direct-v1/val_reports/D0DIRECT_seed42_result.json',
 'AF2DIRECT':'experiments/defectoscafeverde-af2-direct-v1/val_reports/AF2DIRECT_seed42_result.json',
 'DCWCF1':'experiments/defectoscafeverde-cwcf-direct-v1/val_reports/DCWCF1_seed42_result.json',
 'LIFRPF1':'experiments/defectoscafeverde-lif-rpf-v1/val_reports/LIFRPF1_seed42_result.json',
 'RAFC1':'experiments/defectoscafeverde-rafc-v1/val_reports/RAFC1_seed42_result.json'}
CHECKPOINT_REL={
 'D0DIRECT':'experiments/defectoscafeverde-cwcf-direct-v1/D0DIRECT/D0DIRECT_seed42/weights/best.pt',
 'AF2DIRECT':'experiments/defectoscafeverde-af2-direct-v1/AF2DIRECT/AF2DIRECT_seed42/weights/best.pt',
 'DCWCF1':'experiments/defectoscafeverde-cwcf-direct-v1/DCWCF1/DCWCF1_seed42/weights/best.pt',
 'LIFRPF1':'experiments/defectoscafeverde-lif-rpf-v1/LIFRPF1/LIFRPF1_seed42/weights/best.pt',
 'RAFC1':'experiments/defectoscafeverde-rafc-v1/RAFC1/RAFC1_seed42/weights/best.pt'}
RESULTS={name:PROJECT/path for name,path in RESULT_REL.items()}
CHECKPOINTS={name:PROJECT/path for name,path in CHECKPOINT_REL.items()}
missing=[str(path) for path in [*RESULTS.values(),*CHECKPOINTS.values()] if not path.is_file()]
if missing: raise FileNotFoundError('Artefak wajib hilang:\n'+'\n'.join(missing))
archives=sorted({path for root in drive_roots for path in root.rglob(ARCHIVE_NAME) if path.is_file()})
valid_archives=[path for path in archives if path.stat().st_size==EXPECTED_ARCHIVE_BYTES and sha256(path)==EXPECTED_ARCHIVE_SHA]
complete_parts=[]
for scheme,count in (('part16',41),('chunk',11),('part',2)):
    by_parent={}
    for root in drive_roots:
        for path in root.rglob(f'{ARCHIVE_NAME}.{scheme}-*'):
            if path.is_file(): by_parent.setdefault(str(path.parent),{})[path.name]=path
    expected=[f'{ARCHIVE_NAME}.{scheme}-{index:03d}' for index in range(count)]
    for parent,rows in by_parent.items():
        if sorted(rows)==expected and sum(rows[name].stat().st_size for name in expected)==EXPECTED_ARCHIVE_BYTES:
            complete_parts.append((scheme,parent,[rows[name] for name in expected]))
if valid_archives:
    ARCHIVE=valid_archives[0]; ARCHIVE_SOURCE='single archive'
elif complete_parts:
    scheme,parent,parts=complete_parts[0]; ARCHIVE=WORK/ARCHIVE_NAME; ARCHIVE_SOURCE=f'{len(parts)} {scheme} parts dari {parent}'
    with ARCHIVE.open('wb') as target:
        for part in parts:
            with part.open('rb') as source: shutil.copyfileobj(source,target,length=8*1024*1024)
    if sha256(ARCHIVE)!=EXPECTED_ARCHIVE_SHA: raise RuntimeError(f'SHA hasil gabung {scheme} tidak cocok')
else:
    print('Bundle tidak terlihat lewat drive.mount(); memakai fallback Drive API.',flush=True)
    from google.colab import auth
    auth.authenticate_user()
    import google.auth
    from googleapiclient.discovery import build
    from googleapiclient.http import MediaIoBaseDownload
    chunk_ids=['1pAwWlkABja1Epaw1U8Uxre2JorfXwE3P','1UoZkqJbHstFcA9WeXW0KNeisGasqkci6','1kY_xwFOHW17eQQ7x1TDgM6F4ySRyg2gD','1HzW5oJnMOoO7HwWY4Ic-dK8SE-3uy1rH','1ydHf2EqZIJiA_I0udSL68QzbdscRvKtu','1ip1VVRQ6n1NW7Zr5DCU3iajy54Zgsd2B','10cobK2wuX_VsqOd8DfSiwHZcfoXVj4YO','12NpEYi7qfIzzHzItlrw1hFb3DB22N2DQ','15GWk7s87nQGlEBCxe3PU7lxYJsXG1rQY','1miE6MBbwkCme0HX1JF08uDkBDjot-DVZ','1pF9TmGrcK56reCN6zRF4xzIaxtZqD6sk']
    credentials,_=google.auth.default(); service=build('drive','v3',credentials=credentials,cache_discovery=False)
    remote_parts=[]; part_root=WORK/'defectoscafeverde-drive-parts'; part_root.mkdir(exist_ok=True)
    for index,file_id in enumerate(chunk_ids):
        part=part_root/f'{ARCHIVE_NAME}.chunk-{index:03d}'; remote_parts.append(part); expected_size=67108864 if index<10 else 15386112
        if part.is_file() and part.stat().st_size==expected_size: continue
        if part.exists(): part.unlink()
        request=service.files().get_media(fileId=file_id,supportsAllDrives=True)
        with part.open('wb') as stream:
            downloader=MediaIoBaseDownload(stream,request,chunksize=16*1024*1024); done=False
            while not done: _,done=downloader.next_chunk()
        print(f'DOWNLOAD BUNDLE: {index+1}/{len(chunk_ids)}',flush=True)
    if sum(path.stat().st_size for path in remote_parts)!=EXPECTED_ARCHIVE_BYTES: raise RuntimeError('Ukuran chunk Drive API tidak lengkap')
    ARCHIVE=WORK/ARCHIVE_NAME; ARCHIVE_SOURCE='11 Drive API chunks'
    with ARCHIVE.open('wb') as target:
        for part in remote_parts:
            with part.open('rb') as source: shutil.copyfileobj(source,target,length=8*1024*1024)
    if sha256(ARCHIVE)!=EXPECTED_ARCHIVE_SHA: raise RuntimeError('SHA hasil download Drive API tidak cocok')
print('PROJECT:',PROJECT); print('ARCHIVE VALID:',ARCHIVE,'| SOURCE:',ARCHIVE_SOURCE)
print('CHECKPOINTS:',{name:str(path) for name,path in CHECKPOINTS.items()})

In [ ]:
os.chdir(WORK)
if REPO.exists(): shutil.rmtree(REPO)
subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)],check=True,cwd=WORK)
subprocess.run([sys.executable,'-m','pip','install','-q','ultralytics==8.4.96'],check=True,cwd=WORK)
subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps','-e',str(REPO)],check=True,cwd=WORK)
sys.path.insert(0,str(REPO/'src')); importlib.invalidate_caches(); os.chdir(REPO)
import torch
if not torch.cuda.is_available(): raise RuntimeError('Aktifkan GPU Colab; audit lima checkpoint akan lambat di CPU')
print('COMMIT:',subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip())
print('GPU:',torch.cuda.get_device_name(0))

In [ ]:
DATA=WORK/'defectoscafeverde-development-v1'
if DATA.exists(): shutil.rmtree(DATA)
DATA.mkdir(parents=True)
prefix='grouped-physical-v1/'; allowed_files={'data.yaml','grouped_audit.json','grouped_summary.json'}
with tarfile.open(ARCHIVE,'r') as bundle:
    for member in bundle.getmembers():
        name=member.name.replace('\\','/')
        if not name.startswith(prefix): continue
        relative=name[len(prefix):].strip('/')
        if not relative: continue
        first=relative.split('/',1)[0]
        if not (first in {'train','val'} or relative in allowed_files): continue
        target=(DATA/relative).resolve()
        if DATA.resolve() not in target.parents and target!=DATA.resolve(): raise RuntimeError('Unsafe archive path')
        if member.isdir(): target.mkdir(parents=True,exist_ok=True); continue
        target.parent.mkdir(parents=True,exist_ok=True); source=bundle.extractfile(member)
        if source is None: raise RuntimeError(f'Gagal membaca {member.name}')
        with source, target.open('wb') as out: shutil.copyfileobj(source,out)
if (DATA/'test').exists(): raise RuntimeError('TEST TEREXPOSE - STOP')
import yaml
payload=yaml.safe_load((DATA/'data.yaml').read_text()); payload.pop('test',None)
payload['path']=str(DATA); payload['train']='train/images'; payload['val']='val/images'
(DATA/'data.yaml').write_text(yaml.safe_dump(payload,sort_keys=False),encoding='utf-8')
print('TRAIN:',len(list((DATA/'train/images').glob('*'))),'VAL:',len(list((DATA/'val/images').glob('*'))),'TEST:',(DATA/'test').exists())

In [ ]:
from coffee_detector.analysis.defectoscafeverde_error_complementarity import run_error_complementarity_audit
OUT=PROJECT/'experiments/defectoscafeverde-error-complementarity-v1'; OUT.mkdir(parents=True,exist_ok=True)
SUMMARY=OUT/'defectoscafeverde_error_complementarity.json'
RECORDS=OUT/'defectoscafeverde_error_complementarity_targets.jsonl'
result=run_error_complementarity_audit(DATA,RESULTS,CHECKPOINTS,SUMMARY,records_output=RECORDS,device='0')
print('SUMMARY:',SUMMARY); print('TARGET RECORDS:',RECORDS)

In [ ]:
import pandas as pd
final=result['stages']['final_conf0001']
model_table=pd.DataFrame(final['models']).T[['targets','accessible','matched','correct_class','proposal_accessibility','matched_recall','correct_decision_recall','localization_conditioned_class_accuracy']]
display(model_table.style.format({'proposal_accessibility':'{:.2%}','matched_recall':'{:.2%}','correct_decision_recall':'{:.2%}','localization_conditioned_class_accuracy':'{:.2%}'}))
comparison=pd.DataFrame(final['candidate_vs_reference']).T
display(comparison)
class_rows=[]
for name,row in final['per_class'].items():
    class_rows.append({'class':name,'targets':row['targets'],'d0_correct_recall':row['reference_correct_decision_recall'],'union_correct_recall':row['union_correct_decision_recall'],'union_gain':row['union_gain_over_reference'],'rescues':row['rescued_reference_errors_by_model']})
class_table=pd.DataFrame(class_rows).sort_values(['union_gain','class'],ascending=[False,True])
display(class_table.style.format({'d0_correct_recall':'{:.2%}','union_correct_recall':'{:.2%}','union_gain':'{:+.2%}'}))
print('FINAL UNION:',final['union'])
print('WORST CLASS:',result['reference_worst_class_by_ap'])
print('CRITERIA:',result['criteria'])
print('DECISION:',result['decision'])
print('NEXT:',result['next'])
print('TRAINING:',result['training_executed'],'| TEST:',result['test_images_accessed'])
print('Kirim tiga tabel, FINAL UNION, WORST CLASS, CRITERIA, DECISION, dan NEXT. Jangan training atau membuka test.')